## Genetic Classification

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pickle
import random
import seaborn as sns
import tensorflow as tf

from keras.layers import Dense, Dropout, MaxPooling2D, Flatten, Conv2D, BatchNormalization
from keras.models import Sequential
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.inspection import PartialDependenceDisplay
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow import keras
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

### Loading & Preparing the X Datasets

The code begins by loading the merged training, validation, and test datasets stored as pickle files. After loading the datasets, the structure and dimensions of the data are examined to ensure proper understanding. Next, the genetic data columns relevant to Alzheimer's disease prediction are extracted from the merged datasets, and partitioned into their respective three subsets: training, validation, and test. 

Finally, the first few rows of the training dataset along with its shape and column names are printed for verification and validation of data preprocessing steps.

In [ ]:
# loading train, test, and validation datasets
X_train = pd.read_pickle(r"C:\Users\kishe\Documents\Year 3 Jupyter\X_train_merged.pkl")
X_val = pd.read_pickle(r"C:\Users\kishe\Documents\Year 3 Jupyter\X_val_merged.pkl")
X_test  = pd.read_pickle(r"C:\Users\kishe\Documents\Year 3 Jupyter\X_test_merged.pkl")
print(X_train.columns)

# dropping any NaN values that remain
X_train = X_train.dropna()
X_val = X_val.dropna()
X_test = X_test.dropna()
print("Shape of cleaned DataFrame:", X_train.shape)

In [ ]:
# extracting the genetic features from the merged datasets
columns = ['ABCA7', 'ADAM10', 'APOE', 'APP', 'CD2AP', 'CLU', 'LRRK2',
       'PICALM', 'SORL1', 'TREM2']
X_train_genetic = X_train[columns]
X_val_genetic = X_val[columns]
X_test_genetic = X_test[columns]

print(X_train_genetic.shape)
print(X_val_genetic.shape)
print(X_test_genetic.shape)
print(X_train_genetic.head(10))

### Loading & Preparing the Y Labels Datasets

Afterwards, the diagnosis labels are extracted from the merged dataset for the training, validation, and test subsets. These represent the clinical diagnosis of each subject and are stored in separate variables (`y_train`, `y_val`, and `y_test`). The labels are then flattened to 1D arrays and mapped to numerical values. 

A dictionary (`label_mapping`) is defined to map diagnostic categories ("AD" for Alzheimer's disease, "CN" for cognitively normal, and "MCI" for mild cognitive impairment) to numerical values (0, 1, and 2, respectively). The labels in the training, validation, and test sets are then replaced with their corresponding numerical values using list comprehensions and NumPy arrays.

In [ ]:
# extracting the diagnosis labels from the merged dataset
y_train = X_train["Diagnosis"]
y_val = X_val["Diagnosis"]
y_test = X_test["Diagnosis"]

# flattening to 1D array
y_train = y_train.values.ravel()
y_val = y_val.values.ravel() 
y_test = y_test.values.ravel()

print(type(y_train))
print(y_train[:20])
print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

In [ ]:
# encoding the categorical diagnosis labels with numerical mappings
label_mapping = {"AD": 0, "CN": 1, "MCI": 2}
y_train = np.array([label_mapping[label] for label in y_train])
y_val = np.array([label_mapping[label] for label in y_val])
y_test = np.array([label_mapping[label] for label in y_test])

print(type(y_train))
print(y_train[:20])

### Training the GBM Classifier

Code evaluation loop developed with guidance and adapted from: https://github.com/rsinghlab/MADDi/blob/main/training/train_clinical.py

This classifier initiates a Gradient Boosting Classifier model with specific hyperparameters, such as the number of estimators, learning rate, and maximum depth. This model is trained on the genetic features of the training dataset and subsequently used to predict labels for the genetic features in the validation and test datasets. The model's performance is then evaluated using various metrics, including validation accuracy, overall accuracy, precision, recall, and F1 score. Additionally, a confusion matrix is generated to visualize the model's classification performance across different classes. This matrix provides a detailed breakdown of the model's predictions, allowing for a comprehensive assessment of its effectiveness in classifying samples into their respective categories.

In [ ]:
# setting random seeds for reproducibility
def reset_random_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)  
    random.seed(seed)  
    np.random.seed(seed)  

# lists to store evaluation metrics
acc = []
prec = []
rec = []
f1 = []

# generating random seeds for experiments
seeds = [42, 10, 53, 78, 20]

# looping through each seed
for seed in seeds:
    print("Seed:", seed)
    # reset seed
    reset_random_seeds(seed)
    
    # creating the gradient boosting model
    gb_model = GradientBoostingClassifier(n_estimators=1000, 
                                          learning_rate=0.001, 
                                          max_depth=500, 
                                          min_samples_split=115, 
                                          min_samples_leaf=55, 
                                          random_state=seed)

    # fitting the model on training data and predicting labels for test data
    gb_model.fit(X_train_genetic, y_train)
    y_pred = gb_model.predict(X_test_genetic)

    # evaluating the model
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1score = f1_score(y_test, y_pred, average='macro')

    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1score)

    # appending metrics to the lists
    acc.append(accuracy)
    prec.append(precision)
    rec.append(recall)
    f1.append(f1score)

    # printing a confusion matrix
    conf_matrix = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.xlabel('Predicted labels')
    plt.ylabel('True labels')
    plt.title('Confusion Matrix')
    plt.show()

# calculating average and standard deviation for all metrics
print("Average Accuracy:", np.mean(acc))
print("Average Precision:", np.mean(precision))
print("Average Recall:", np.mean(recall))
print("Average F1 Score:", np.mean(f1))
print("Standard Deviation Accuracy:", np.std(acc))
print("Standard Deviation Precision:", np.std(precision))
print("Standard Deviation Recall:", np.std(recall))
print("Standard Deviation F1 Score:", np.std(f1))

## Explainable AI

### PDP

Partial dependence plots are computed for the specified features, and the plots are displayed using `PartialDependenceDisplay.from_estimator()` function from the sklearn.inspection module. The plots show how the predicted outcome (probability of the target class) changes as each specified feature varies while holding other features constant. This allows for the visualization of the marginal effect of each feature on the model's predictions.

In [ ]:
# specifying the target class for which to compute partial dependence
target_class = 0 

feature_names = list(X_train_genetic)

# computing partial dependence and plotting it
PartialDependenceDisplay.from_estimator(gb_model, 
                        X_train_genetic, 
                        features=[1, 7, 9, 4],
                        feature_names=feature_names, 
                        grid_resolution=100,
                        target=target_class)

plt.tight_layout()
plt.figure(figsize=(4, 4))
plt.show()

In [ ]:
# specifying the target class for which to compute partial dependence
target_class = 1 

feature_names = list(X_train_genetic)

# increasing plot size
plt.figure(figsize=(10, 10))

# computing partial dependence and plotting it
PartialDependenceDisplay.from_estimator(gb_model, 
                        X_train_genetic, 
                        features=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
                            # (0, 1), (2, 3), (4, 5), (6, 7), (8, 9)],
                        feature_names=feature_names, 
                        grid_resolution=50,
                        target=target_class,
                        kind='both', centered=True)

plt.tight_layout()
plt.show()

### Feature Relative Importance
This code box generates a horizontal bar chart to visualise the feature importances of `gb_model`. It first extracts the importances and sorts them. Then, it plots the sorted importances against the corresponding feature names. The length of each bar represents the feature's relative importance. This visualisation helps identify the most influential features in the model, aiding in feature selection and interpretation.

In [ ]:
importances = gb_model.feature_importances_
indices = np.argsort(importances)
features = X_train_genetic.columns
plt.title('Feature Importances')
plt.barh(range(len(indices)), importances[indices], color='g', align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

## Yellowbrick
From: https://www.scikit-yb.org/en/latest/

Using the Yellowbrick library to create a visualization called FeatureCorrelation, specifically using mutual information as the correlation method (`method='mutual_info-classification'`). Mutual information measures the statistical dependency between two variables by quantifying the amount of information obtained about one variable through observing the other. In the context of feature correlation, mutual information helps determine the strength of the relationship between each feature and the target variable (`y_train`). 

The visualisation displays a graph, where higher values indicate stronger associations between features and the target, aiding in feature selection and understanding the predictive power of each feature.

In [ ]:
from yellowbrick.target import FeatureCorrelation

feature_names = list(X_train_genetic.columns)

visualizer = FeatureCorrelation(
    method='mutual_info-classification', feature_names=feature_names, sort=True
)

visualizer.fit(X_train_genetic, y_train)       
visualizer.show()      

In [ ]:
from yellowbrick.classifier import ClassificationReport

visualizer = ClassificationReport(gb_model)

visualizer.fit(X_train_genetic, y_train)
visualizer.score(X_test_genetic, y_test)
visualizer.show()

This code generates a `Rank2D` visualisation using Pearson correlation as the algorithm (`algorithm="pearson"`). Pearson correlation measures the linear relationship between two variables, providing a value between -1 and 1, where 1 indicates a perfect positive linear relationship, -1 indicates a perfect negative linear relationship, and 0 indicates no linear relationship. 

Pearson correlation evaluates the pairwise correlation between features in `X_train_genetic`. The resulting matrix displays the strength and direction of linear relationships between pairs of features, helping identify potentially redundant or highly correlated features.

In [ ]:
from yellowbrick.features import Rank2D

visualizer = Rank2D(algorithm="pearson")
visualizer.fit_transform(X_train_genetic)
visualizer.show()

Creating a Joint Plot visualisation using the `JointPlotVisualizer` class. This explores the relationship between two specific features, from the dataset. The Joint Plot displays both the univariate distributions of each feature along the corresponding axes and the bivariate relationship between them through a scatter plot in the center. Additionally, the visualisation includes a linear regression line to indicate the trend or correlation between the two features.

In [ ]:
from yellowbrick.features import JointPlotVisualizer

visualizer = JointPlotVisualizer(columns=['CD2AP', 'APP'])
visualizer.fit_transform(X_train_genetic, y_train)
visualizer.show()

In [ ]:
visualizer = JointPlotVisualizer(columns=['APOE', 'APP'])
visualizer.fit_transform(X_train_genetic, y_train)
visualizer.show()

### Shapash
From: https://shapash.readthedocs.io/en/latest/

Using SmartExplainer from the Shapash library. After initializing SmartExplainer with the trained gradient boosting model (`gb_model`), it's compiled with relevant data: input features, model predictions (`y_pred_series`), and actual target values (`y_test_series`). 

Visualisations are then generated. First, `xpl.plot.features_importance()` creates a plot illustrating feature importance, revealing which variables have the most influence on model predictions. Second, `xpl.plot.contribution_plot("APOE")` produces a contribution plot for the "APOE" (or other gene) feature, detailing its impact on model predictions. 

In [ ]:
from shapash import SmartExplainer

xpl = SmartExplainer(model=gb_model)

y_pred_series = pd.Series(y_pred)
y_test_series = pd.Series(y_test)

X_test_genetic.index = range(len(X_test_genetic))
y_pred_series.index = range(len(y_pred_series))

xpl.compile(x=X_test_genetic,
 y_pred=y_pred_series,
 y_target=y_test_series,
 )

In [ ]:
xpl.plot.features_importance()

In [ ]:
xpl.plot.contribution_plot("TREM2")

In [ ]:
xpl.plot.contribution_plot("APP")

In [ ]:
xpl.plot.contribution_plot("SORL1")